# Persian Text Generation — Inference

Generate Persian text from a seed word using all four trained models:
**LSTM**, **GPT-style Transformer**, **Dilated Causal CNN**, and **SVM**.

## Requirements
Before running, make sure the following are saved on your Google Drive:

| Model | Path on Drive |
|-------|---------------|
| LSTM | `Persian_Text_Generation/models/lstm` |
| Transformer | `Persian_Text_Generation/models/transformer` |
| CNN | `Persian_Text_Generation/models/cnn` |
| SVM | `Persian_Text_Generation/models/svm` |

Each folder must contain `model.pt` and a `tokenizer/` subfolder.
The SVM folder must also contain `svm_meta.json`.

## 0. Setup

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'hazm', 'scikit-learn', 'joblib'], capture_output=True)

import os, re, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
from transformers import AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive'

## 2. Generation Settings

In [ ]:
# ── Prompt ───────────────────────────────────────────────────────────────
SEED_TEXT      = 'امروز'   # <-- change this to any Persian word or phrase

# ── Shared generation parameters ─────────────────────────────────────────
MAX_NEW_TOKENS = 25        # maximum tokens to generate
TEMPERATURE    = 0.9       # higher = more diverse, lower = more focused
TOP_P          = 0.9       # nucleus sampling threshold
REP_PENALTY    = 1.5       # penalise repeated tokens

# ── Model paths on Drive ─────────────────────────────────────────────────
LSTM_DIR   = os.path.join(DRIVE_BASE, 'Persian_Text_Generation/models/lstm')
GPT_DIR    = os.path.join(DRIVE_BASE, 'Persian_Text_Generation/models/transformer')
CNN_DIR    = os.path.join(DRIVE_BASE, 'Persian_Text_Generation/models/cnn')
SVM_DIR    = os.path.join(DRIVE_BASE, 'Persian_Text_Generation/models/svm')

# ── Architecture constants (must match training config) ───────────────────
MAX_LENGTH   = 32

LSTM_EMBED   = 128
LSTM_HIDDEN  = 256
LSTM_LAYERS  = 2
LSTM_DROPOUT = 0.2

GPT_EMBED    = 256
GPT_HEADS    = 8
GPT_FF       = 512
GPT_LAYERS   = 4
GPT_DROPOUT  = 0.1

CNN_EMBED    = 256
CNN_CHANNELS = 256
CNN_KERNEL   = 3
CNN_BLOCKS   = 4
CNN_DROPOUT  = 0.1

SVM_CONTEXT  = 5

print('Settings loaded.')
print(f'Seed text: {SEED_TEXT!r}')

## 3. Load Tokenizer

In [ ]:
# All four models share the same tokenizer
tokenizer = AutoTokenizer.from_pretrained('HooshvareLab/gpt2-fa')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

VOCAB_SIZE = len(tokenizer)
PAD_ID     = tokenizer.pad_token_id
EOS_ID     = tokenizer.eos_token_id
print(f'Vocab size : {VOCAB_SIZE:,}')

## 4. Model Definitions

In [ ]:
# ── LSTM ─────────────────────────────────────────────────────────────────
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_ID)
        self.lstm    = nn.LSTM(embed_size, hidden_size, num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.ln      = nn.LayerNorm(hidden_size)
        self.fc      = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        emb         = self.dropout(self.embed(x))
        out, hidden = self.lstm(emb, hidden)
        return self.fc(self.dropout(self.ln(out))), hidden


# ── GPT-style Transformer ─────────────────────────────────────────────────
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_heads, ff_dim,
                 n_layers, max_len, dropout):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        self.drop    = nn.Dropout(dropout)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            enc_layer, num_layers=n_layers, enable_nested_tensor=False)
        self.ln_f = nn.LayerNorm(embed_dim)
        self.fc   = nn.Linear(embed_dim, vocab_size, bias=False)
        self.fc.weight = self.tok_emb.weight  # weight tying

    def _causal_mask(self, T, dev):
        return torch.triu(torch.full((T, T), float('-inf'), device=dev), diagonal=1)

    def forward(self, x):
        B, T  = x.shape
        pos   = torch.arange(T, device=x.device).unsqueeze(0)
        emb   = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        mask  = self._causal_mask(T, x.device)
        pad_m = torch.zeros_like(x, dtype=torch.float).masked_fill(x == PAD_ID, float('-inf'))
        out   = self.transformer(emb, mask=mask, src_key_padding_mask=pad_m)
        return self.fc(self.ln_f(out))


# ── Dilated Causal CNN ────────────────────────────────────────────────────
class DilatedCausalBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout):
        super().__init__()
        self.pad  = (kernel_size - 1) * dilation  # left-only causal padding
        self.conv = nn.Conv1d(channels, channels, kernel_size=kernel_size,
                              dilation=dilation, padding=0)
        self.ln      = nn.LayerNorm(channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.conv(F.pad(x, (self.pad, 0)))
        x = x.permute(0, 2, 1)
        x = self.dropout(F.gelu(self.ln(x)))
        return x.permute(0, 2, 1) + residual


class DilatedCausalCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_channels,
                 kernel_size, n_blocks, dropout):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.input_proj = nn.Linear(embed_dim, n_channels)
        self.blocks     = nn.ModuleList([
            DilatedCausalBlock(n_channels, kernel_size, 2**i, dropout)
            for i in range(n_blocks)
        ])
        self.ln_out = nn.LayerNorm(n_channels)
        self.fc     = nn.Linear(n_channels, vocab_size)

    def forward(self, x):
        out = self.input_proj(self.embedding(x)).permute(0, 2, 1)
        for block in self.blocks:
            out = block(out)
        return self.fc(self.ln_out(out.permute(0, 2, 1)))

print('Model classes defined.')

## 5. Load Trained Weights

In [ ]:
def load_model(model, directory):
    path = os.path.join(directory, 'model.pt')
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model

# LSTM
lstm_model = load_model(
    LSTMLanguageModel(VOCAB_SIZE, LSTM_EMBED, LSTM_HIDDEN, LSTM_LAYERS, LSTM_DROPOUT).to(device),
    LSTM_DIR
)
print('LSTM     loaded')

# GPT Transformer
gpt_model = load_model(
    GPTLanguageModel(VOCAB_SIZE, GPT_EMBED, GPT_HEADS, GPT_FF,
                     GPT_LAYERS, MAX_LENGTH, GPT_DROPOUT).to(device),
    GPT_DIR
)
print('GPT      loaded')

# Dilated Causal CNN
cnn_model = load_model(
    DilatedCausalCNN(VOCAB_SIZE, CNN_EMBED, CNN_CHANNELS,
                     CNN_KERNEL, CNN_BLOCKS, CNN_DROPOUT).to(device),
    CNN_DIR
)
print('CNN      loaded')

# SVM pipeline
svm_pipeline = joblib.load(os.path.join(SVM_DIR, 'svm_pipeline.joblib'))
with open(os.path.join(SVM_DIR, 'svm_meta.json')) as f:
    svm_meta    = json.load(f)
svm_classes  = svm_pipeline.classes_
# handle both int and str keys in cls_to_id
cls_to_id    = {int(k): int(v) for k, v in svm_meta['cls_to_id'].items()}
print('SVM      loaded')

## 6. Sampling Utilities

In [ ]:
def apply_rep_penalty_torch(logits, generated_ids, penalty):
    """Divide logit scores of already-generated tokens by penalty."""
    for tok_id in set(generated_ids):
        if logits[tok_id] > 0:
            logits[tok_id] /= penalty
        else:
            logits[tok_id] *= penalty
    return logits


def nucleus_sample_torch(logits, top_p):
    """Nucleus (top-p) sampling on a 1-D logit tensor."""
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_logits[cum - F.softmax(sorted_logits, dim=-1) > top_p] = float('-inf')
    logits = torch.zeros_like(logits).scatter_(0, sorted_idx, sorted_logits)
    return torch.multinomial(F.softmax(logits, dim=-1), 1).item()


def cls_lookup(cls_str, cls_to_id):
    """Convert SVM class string to token ID; handles int or str keys."""
    v = cls_to_id.get(int(cls_str))
    if v is None:
        v = cls_to_id.get(str(cls_str))
    return int(v) if v is not None else None

## 7. Generation Functions

In [ ]:
def generate_lstm(seed_text):
    ids    = tokenizer.encode(seed_text, add_special_tokens=False) or [EOS_ID]
    hidden = None
    # warm up hidden state on the prompt
    with torch.no_grad():
        _, hidden = lstm_model(torch.tensor([ids], dtype=torch.long, device=device), hidden)
    last_id = torch.tensor([[ids[-1]]], dtype=torch.long, device=device)
    generated = []
    for _ in range(MAX_NEW_TOKENS):
        with torch.no_grad():
            logits, hidden = lstm_model(last_id, hidden)
        next_logits = apply_rep_penalty_torch(logits[0, -1] / TEMPERATURE,
                                              ids + generated, REP_PENALTY)
        next_id = nucleus_sample_torch(next_logits, TOP_P)
        if next_id == EOS_ID:
            break
        generated.append(next_id)
        last_id = torch.tensor([[next_id]], dtype=torch.long, device=device)
    return tokenizer.decode(ids + generated, skip_special_tokens=True)


def generate_gpt(seed_text):
    ids = tokenizer.encode(seed_text, add_special_tokens=False) or [EOS_ID]
    for _ in range(MAX_NEW_TOKENS):
        inp = torch.tensor([ids[-MAX_LENGTH:]], dtype=torch.long, device=device)
        with torch.no_grad():
            logits = gpt_model(inp)
        next_logits = apply_rep_penalty_torch(logits[0, -1] / TEMPERATURE,
                                              ids, REP_PENALTY)
        next_id = nucleus_sample_torch(next_logits, TOP_P)
        if next_id == EOS_ID:
            break
        ids.append(next_id)
    return tokenizer.decode(ids, skip_special_tokens=True)


def generate_cnn(seed_text):
    ids = tokenizer.encode(seed_text, add_special_tokens=False) or [EOS_ID]
    for _ in range(MAX_NEW_TOKENS):
        inp = torch.tensor([ids[-MAX_LENGTH:]], dtype=torch.long, device=device)
        with torch.no_grad():
            logits = cnn_model(inp)
        next_logits = apply_rep_penalty_torch(logits[0, -1] / TEMPERATURE,
                                              ids, REP_PENALTY)
        next_id = nucleus_sample_torch(next_logits, TOP_P)
        if next_id == EOS_ID:
            break
        ids.append(next_id)
    return tokenizer.decode(ids, skip_special_tokens=True)


def generate_svm(seed_text):
    ids = tokenizer.encode(seed_text, add_special_tokens=False) or [EOS_ID]
    for _ in range(MAX_NEW_TOKENS):
        ctx_str = ' '.join(str(t) for t in ids[-SVM_CONTEXT:])
        proba   = svm_pipeline.predict_proba([ctx_str])[0]
        # temperature scaling
        log_p   = np.log(proba + 1e-9) / TEMPERATURE
        log_p  -= log_p.max()
        p       = np.exp(log_p)
        # repetition penalty
        for i, cls in enumerate(svm_classes):
            tok_id = cls_lookup(cls, cls_to_id)
            if tok_id is not None and tok_id in ids:
                p[i] = max(0.0, p[i] / REP_PENALTY)
        # nucleus sampling
        sorted_idx  = np.argsort(p)[::-1]
        cum_p       = np.cumsum(p[sorted_idx])
        cutoff      = np.searchsorted(cum_p, TOP_P) + 1
        nucleus_idx = sorted_idx[:cutoff]
        nucleus_p   = p[nucleus_idx]
        total       = nucleus_p.sum()
        if total < 1e-9:
            break
        nucleus_p  /= total
        chosen_cls  = svm_classes[nucleus_idx[np.random.choice(len(nucleus_idx), p=nucleus_p)]]
        next_id     = cls_lookup(chosen_cls, cls_to_id)
        if next_id is None or next_id == EOS_ID:
            break
        ids.append(next_id)
    return tokenizer.decode(ids, skip_special_tokens=True)

print('Generation functions ready.')

## 8. Run Inference

Change `SEED_TEXT` in **cell 2** and re-run this cell to generate new outputs.

In [ ]:
results = {
    'SVM'        : generate_svm(SEED_TEXT),
    'CNN'        : generate_cnn(SEED_TEXT),
    'LSTM'       : generate_lstm(SEED_TEXT),
    'Transformer': generate_gpt(SEED_TEXT),    
}

print('\n' + '='*65)
print(f'  Seed: {SEED_TEXT!r}')
print('='*65)
for model_name, output in results.items():
    print(f'  {model_name:<12}: {output}')
print('='*65)